# CNN-LSTM 融合的消融实验

## 研究问题

在目前固定的 CNN + 单向 LSTM 等权软投票中，两个模型分别对验证集分类结果有什么影响？完整融合是否比去掉其中一个模型后更好？

## A1：融合组成消融

- 完整方案：CNN 与单向 LSTM 的正类概率各占 0.5。
- 去掉 LSTM：CNN 的权重重新归一为 1，等价于 CNN 单模型。
- 去掉 CNN：LSTM 的权重重新归一为 1，等价于 LSTM 单模型。

控制验证集及样本顺序、预处理、原模型检查点、预测概率和分类阈值；统一使用种子 42 和 123 两对现有检查点，每对模型使用相同的种子。概率大于 0.5 时预测正类，等于 0.5 时预测负类。评价 Accuracy、Precision、Recall、F1 和纠正／新引入错误。

这里移除的是融合时的预测分量，不重新训练剩余模型，也不声称测量了组件的独立因果效应。由于相同模型和验证结果此前已经被观察，本组属于回顾性的探索，不是预先注册的新确认性实验。两个种子共用验证集，不能视为独立数据集。


In [ ]:
import hashlib
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.lstm import LSTMClassifier
from sentiment_analysis.paths import CHECKPOINTS_DIR, CLEANED_DIR, TABLES_DIR
from sentiment_analysis.sequence_data import SentimentDataset, TOKEN_PATTERN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
validation_path = CLEANED_DIR / "val.csv"
validation_data = pd.read_csv(validation_path)
probabilities = pd.read_csv(TABLES_DIR / "textcnn_bilstm_validation_probabilities.csv")
with (TABLES_DIR / "textcnn_bilstm_ensemble_metadata.json").open(encoding="utf-8") as file:
    metadata = json.load(file)

if hashlib.sha256(validation_path.read_bytes()).hexdigest() != metadata["validation_sha256"]:
    raise ValueError("Validation data does not match the saved probabilities.")
if len(validation_data) != metadata["validation_rows"]:
    raise ValueError("Validation data size does not match the saved probabilities.")
print(f"Device: {device} | Validation: {len(validation_data):,}")


In [ ]:
# 加载单向 LSTM 检查点，核对与对应 CNN 使用相同的序列规则
checkpoint_paths = {
    42: CHECKPOINTS_DIR / "lstm_best.pt",
    123: CHECKPOINTS_DIR / "lstm_unidirectional_seed123_best.pt",
}
predictions = {}

for seed, path in checkpoint_paths.items():
    saved = probabilities[probabilities["SEED"] == seed].sort_values("VALIDATION ROW")
    if (saved["VALIDATION ROW"].tolist() != list(range(len(validation_data)))
            or saved["LABEL"].tolist() != validation_data["label"].tolist()):
        raise ValueError("Saved probabilities do not align with validation rows.")

    checkpoint = torch.load(path, map_location=device, weights_only=True)
    cnn_info = metadata["checkpoint_files"][str(seed)]
    cnn_path = CHECKPOINTS_DIR / cnn_info["textcnn_file"]
    bi_path = CHECKPOINTS_DIR / cnn_info["bilstm_file"]
    if hashlib.sha256(cnn_path.read_bytes()).hexdigest() != cnn_info["textcnn_sha256"]:
        raise ValueError("TextCNN checkpoint does not match the saved probabilities.")
    if hashlib.sha256(bi_path.read_bytes()).hexdigest() != cnn_info["bilstm_sha256"]:
        raise ValueError("BiLSTM checkpoint does not match the saved probabilities.")
    cnn_checkpoint = torch.load(cnn_path, map_location=device, weights_only=True)
    if (checkpoint["vocabulary"] != cnn_checkpoint["vocabulary"]
            or checkpoint["max_length"] != cnn_checkpoint["max_length"]
            or checkpoint["token_pattern"] != TOKEN_PATTERN.pattern
            or checkpoint["token_pattern"] != cnn_checkpoint["token_pattern"]
            or not checkpoint["lowercase"] or not cnn_checkpoint["lowercase"]
            or checkpoint["random_seed"] != seed):
        raise ValueError("Checkpoint sequence processing or seed does not match.")

    dataset = SentimentDataset(validation_data, checkpoint["vocabulary"], checkpoint["max_length"])
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    model = LSTMClassifier(**checkpoint["model_config"]).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    labels = []
    lstm_probabilities = []
    with torch.no_grad():
        for token_ids, lengths, batch_labels in loader:
            logits = model(token_ids.to(device), lengths)
            labels.extend(batch_labels.tolist())
            lstm_probabilities.extend(torch.softmax(logits, dim=1)[:, 1].cpu().tolist())

    if labels != validation_data["label"].tolist():
        raise ValueError("LSTM predictions do not align with validation rows.")
    predictions[seed] = {
        "label": np.array(labels),
        "cnn": saved["TEXTCNN PROBABILITY"].to_numpy(),
        "uni": np.array(lstm_probabilities),
        "bi": saved["BILSTM PROBABILITY"].to_numpy(),
    }
    print(f"Seed {seed}: single-direction LSTM epoch {checkpoint['epoch']}")


In [ ]:
# 比较完整融合及去掉每个预测分量后的结果
rows = []
for seed, data in predictions.items():
    label = data["label"]
    configurations = [
        ("Full ensemble", 0.5, 0.5),
        ("Without LSTM", 1.0, 0.0),
        ("Without CNN", 0.0, 1.0),
    ]
    for name, cnn_weight, lstm_weight in configurations:
        probability = cnn_weight * data["cnn"] + lstm_weight * data["uni"]
        prediction = (probability > 0.5).astype(int)
        rows.append({
            "SEED": seed,
            "CONFIGURATION": name,
            "CNN WEIGHT": cnn_weight,
            "LSTM WEIGHT": lstm_weight,
            **classification_metrics(label, prediction),
        })

ablation_results = pd.DataFrame(rows)
display(ablation_results.round(4))
ablation_results.to_csv(TABLES_DIR / "ensemble_component_ablation_validation.csv", index=False)


In [ ]:
# 对每个种子计算完整融合相对单模型的 F1 变化及预测变化
rows = []
for seed, data in predictions.items():
    label = data["label"]
    full_prediction = ((data["cnn"] + data["uni"]) / 2 > 0.5).astype(int)
    full_correct = full_prediction == label
    full_result = ablation_results[
        (ablation_results["SEED"] == seed) & (ablation_results["CONFIGURATION"] == "Full ensemble")
    ].iloc[0]

    for name, key in [("Without LSTM", "cnn"), ("Without CNN", "uni")]:
        baseline_prediction = (data[key] > 0.5).astype(int)
        baseline_correct = baseline_prediction == label
        baseline_result = ablation_results[
            (ablation_results["SEED"] == seed) & (ablation_results["CONFIGURATION"] == name)
        ].iloc[0]
        rows.append({
            "SEED": seed,
            "COMPARATOR": name,
            "FULL F1 MINUS COMPARATOR": full_result["F1"] - baseline_result["F1"],
            "CORRECTED": int((~baseline_correct & full_correct).sum()),
            "NEW ERRORS": int((baseline_correct & ~full_correct).sum()),
        })

ablation_changes = pd.DataFrame(rows)
display(ablation_changes.round(4))
ablation_changes.to_csv(TABLES_DIR / "ensemble_component_ablation_changes_validation.csv", index=False)


## A1 观察

- 两个种子下，完整融合的验证 F1 均高于单独使用 CNN 或 LSTM；相对 CNN 分别高出约 0.0143 和 0.0148。
- 相对 CNN，种子 42 的融合纠正 117 条并引入 75 条错误；种子 123 纠正 140 条并引入 101 条错误。
- 这复用了此前已经观察过的模型与验证集，结果说明当前组合的表现，并不能证明任何分量必然带来提升。


## 后续待设计的模型内部消融

A1 只检验融合阶段，不检验模型内部设计。后续可单独研究 TextCNN 的 Dropout：固定数据、词表、卷积核、训练配置、早停规则和配对随机种子，从头分别训练 Dropout 0.5 与 0 的模型。应记录各次训练曲线、最佳轮次和验证指标，再观察波动；不能仅修改已经训练好的模型的推理开关来代替重新训练。

这是一项待执行的方案，训练成本及最终配置应在实验前明确。测试集已经查看，不作为后续消融方案的调参或筛选依据。
